# scarlatti-doodle project!

## Zip 파일 메모리에 로딩
#### this code block was written using AI

In [2]:
import io
import os
import zipfile
import tempfile
import partitura as pt
from music21 import midi, environment

# music21 임시 디렉토리 억까 방지 설정
try:
    environment.Environment()['directoryScratch'] = '/tmp'
except:
    pass

def mount_scarlatti():
    """scarlatti.zip을 가상 메모리에 마운트하고, 파일 목록을 반환합니다."""
    ZIP_FILE_PATH = "original_midi.zip"
    
    if not os.path.exists(ZIP_FILE_PATH):
        raise FileNotFoundError(f"⚠️ '{ZIP_FILE_PATH}' 파일이 없습니다. 왼쪽 탐색기에 업로드해 주세요!")
        
    print("🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...")
    
    with open(ZIP_FILE_PATH, "rb") as f:
        zip_buffer = io.BytesIO(f.read())
        
    archive = zipfile.ZipFile(zip_buffer)
    midi_files = [f for f in archive.namelist() if f.lower().endswith(('.mid', '.midi'))]
    print(f"📦 마운트 완료! 총 {len(midi_files)}개의 가상 미디 파일 준비 완료.")
    
    return archive, midi_files

def load_midi_from_virtual_folder(archive, file_path, target_type="partitura"):
    """
    [에러 수정 완료] 가상 폴더에서 데이터를 읽어 지정한 라이브러리 객체로 변환합니다.
    BytesIO 타입 제한 억까를 우회하기 위해 OS 임시 파일 핸들러를 안전하게 사용합니다.
    """
    # 1. 압축 파일 내부에서 순수 바이트 데이터 추출
    midi_raw_bytes = archive.read(file_path)
    
    # 2. 파르티투라 객체로 변환할 때
    if target_type == "partitura":
        # 💡 [핵심 우회] BytesIO를 거부하므로, RAM처럼 작동하는 임시 파일을 시스템에 잠깐 썼다 지웁니다.
        # 디스크에 영구 저장되지 않고 함수가 끝나면 메모리에서 자동 소멸해서 속도가 엄청 빠릅니다!
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mid") as tmp_file:
            tmp_file.write(midi_raw_bytes)
            tmp_file_path = tmp_file.name
        
        try:
            # 안전하게 문자열 경로(PathLike)로 인식시켜서 에러 타파!
            performance = pt.load_performance_midi(tmp_file_path)[0]
        finally:
            # 처리가 끝나면 임시 파일 흔적 지우기
            if os.path.exists(tmp_file_path):
                os.remove(tmp_file_path)
                
        return performance
        
    # 3. 뮤직21 객체로 변환할 때 (기존과 동일, 잘 작동함)
    elif target_type == "music21":
        mf = midi.MidiFile()
        mf.readstr(midi_raw_bytes)
        m21_score = midi.translate.midiFileToStream(mf)
        return m21_score
        
    else:
        raise ValueError("⚠️ target_type은 'partitura' 또는 'music21'만 가능합니다!")

# 🔥 [실행] 가상 폴더 연결하기
scarlatti_folder, file_list = mount_scarlatti()
print("앞으로 scarlatti_folder, file_list에 접근해서 사용!")

/home/codespace/.local/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...
📦 마운트 완료! 총 555개의 가상 미디 파일 준비 완료.
앞으로 scarlatti_folder, file_list에 접근해서 사용!


## 전처리 함수들 준비

In [ ]:
from music21 import *
import partitura as pt
import random
import numpy as np



# 흐름
# 미디파일 -> 퀀타이즈 -> 모든조로 전조 -> 렌덤으로 마디 분리 -> 오른손 왼손 분리 -> 오른손 데이터는 마스킹 -> 마스킹한 오른손 데이터는 인풋으로, 온전한 오른손과 왼손 데이터는 아웃풋으로 넘파이 어레이로 전환해서 ai모델 학습용 리스트에 저장

def Score_Quantize(partituraPerformance: pt.performance.PerformedPart):
    '''25ms 단위로 음들 퀀타이즈해서 리턴하는 함수'''
    note_array = partituraPerformance.note_array().copy()
    for i in range(len(note_array)):
        note_array[i]['onset_sec'] = round(note_array[i]['onset_sec'] * 1000 / 25) * 25 / 1000
        note_array[i]['duration_sec'] = max(0.025, round(note_array[i]['duration_sec'] * 1000 / 25) * 25 / 1000)

    return pt.performance.PerformedPart.from_note_array(note_array)



def Score_TrebleBassSeparation_Partitura(partituraPerformance: pt.performance.PerformedPart, splitPoint: int = 60):
    '''높은음자리표 낮은음자리표 분리해서 리턴하는 함수'''
    performance = partituraPerformance
    note_array = performance.note_array()
    
    split_pitch = splitPoint
    
    treble_mask = note_array['pitch'] >= split_pitch
    bass_mask = note_array['pitch'] < split_pitch
    
    treble_note_array = note_array[treble_mask] # 이거는 단순 마스크! 프린트하면 true와 false 로 이루어진 어레이가 나옴!
    bass_note_array = note_array[bass_mask]
    
    # NumPy 배열을 partitura가 저장할 수 있는 PerformedPart 객체 상자에 다시 담아줍니다.
    treble_part = pt.performance.PerformedPart.from_note_array(treble_note_array)
    bass_part = pt.performance.PerformedPart.from_note_array(bass_note_array)
    
    return treble_part, bass_part


def Score_TransposeToAllKeys(partituraPerformance : pt.performance.PerformedPart, music21Score : stream.Score):
    '''스코어 파일을 모든 조로 전조해서 리턴'''
    transposedPerformances = []
    num = PredictedKeyToNum(music21Score=music21Score)

    note_array = partituraPerformance.note_array().copy()
    note_array["pitch"] -= num

    for _ in range(0, 12):
        transposedPerformances.append(pt.performance.PerformedPart.from_note_array(note_array.copy())) # 제미나이가 넣으라고 해서 넣었는거인데 추후에 확인해보기
        note_array["pitch"] += 1

    return transposedPerformances

def PredictedKeyToNum(music21Score : stream.Score):
    KEY_TO_NUM = {
    'C': 0, 'C#': 1, 'D': 2, 'D#': 3, 'E': 4, 'F': 5,
    'F#': 6, 'G': 7, 'G#': 8, 'A': 9, 'A#': 10, 'B': 11,
    'Db': 1, 'Eb': 3, 'Gb': 6, 'Ab': 8, 'Bb': 10
    }

    predicted_key = str(music21Score.analyze('key').tonic.name)
    num = KEY_TO_NUM[predicted_key.upper().replace('-', 'b')]

    return num


def Score_SliceByMeasures(partituraPerformance : pt.performance.PerformedPart):
    '''스코어 파일을 30초 길이만큼 나눠서 리턴'''
    # 미디를 30초 단위로 나눔, 0에서 2 사이의 마디만큼 겹침, 남은 마디가 부족하면 겹쳐서라도 16마디 맟춤
    slicedPerformances = []
    note_array = partituraPerformance.note_array().copy()
    maxSec = (note_array['onset_sec'] + note_array['duration_sec']).max()
    
    endSec = 30.0

    while(endSec <= maxSec):

        mask = ((endSec - 30.0) <= note_array['onset_sec']) & (note_array['onset_sec'] <= endSec)
        slicedPerformance = note_array[mask].copy() # 주어진 범위 안에 들어가는 음들만 정리

        slicedPerformanceMask = (slicedPerformance['onset_sec'] + slicedPerformance['duration_sec']) > endSec # 범위 안 음들 중 끝 범위 넘어가는 것들 찾아내기
        slicedPerformance['duration_sec'][slicedPerformanceMask] = endSec - slicedPerformance['onset_sec'][slicedPerformanceMask] # 끝 범위 넘어가는 음들은 길이 조정
        slicedPerformance['onset_sec'] -= (endSec - 30.0) # 시작점 0으로 맞춤

        slicedPerformances.append(pt.performance.PerformedPart.from_note_array(slicedPerformance)) # 잘라낸 음들을 partitura 객체로 변환해서 리스트에 추가

        if(maxSec == endSec):
            break
        elif(maxSec - endSec < 30.0): 
            endSec = maxSec
        else:
            endSec = endSec + 30.0 - random.randint(0, 15)
    
    return slicedPerformances


def Score_MaskNotes(partituraTrebblePerformance : pt.performance.PerformedPart):
    '''멜로디 데이터에서 일부 데이터들 마스킹해서 리턴'''
    note_array = partituraTrebblePerformance.note_array().copy()
    survived_note_array = []

    # TODO 그대로두거나 없애거나 늘이거나 줄이는 비율 다른 변형들 있게 코드 수정하기!
    for i in range(0, len(note_array)):
        randNum = random.randint(1, 10)
        if(randNum <= 4): # 그대로두기
            survived_note_array.append(note_array[i])
        elif(randNum <= 8): # 없애기
            continue
        elif(randNum <= 9): # 피치 변형하기
            note_array[i]['pitch'] = min(127, max(0, int(note_array[i]['pitch'] + random.randint(-12, 12))))
            survived_note_array.append(note_array[i])
        else: # 늘이거나 줄이기
            note_array[i]['onset_sec'] = max(round(random.uniform(-0.2, 0.2),6) + note_array[i]['onset_sec'], 0.000001)
            note_array[i]['duration_sec'] = max(round(random.uniform(-0.2, 0.2),6) + note_array[i]['duration_sec'], 0.05)
            survived_note_array.append(note_array[i])

    # onset_sec(x[0])을 기준으로 시간순 정렬!
    survived_note_array.sort(key=lambda x: x['onset_sec'])

    if(len(survived_note_array) == 0):
        return Score_MaskNotes(partituraTrebblePerformance=partituraTrebblePerformance)

    return pt.performance.PerformedPart.from_note_array(np.array(survived_note_array, dtype=note_array.dtype))

def ScoreToDataset(partituraPerformance : pt.performance.PerformedPart):
    '''스코어 파일을 ai 학습용 데이터셋으로 변환해서 리턴'''
    array = np.zeros((128, 40 * 30))
    note_array = partituraPerformance.note_array().copy()

    for i in range(len(note_array)):
        pitch = note_array[i]['pitch']
        startPoint = int(round(note_array[i]['onset_sec'] * 1000 / 25))
        endPoint = int(round((note_array[i]['onset_sec'] + note_array[i]['duration_sec']) * 1000 / 25))

        if(startPoint >= 1200):
            continue
        endPoint = min(endPoint, 1200)

        array[pitch, startPoint] = 2 # 음이 시작하는 부분은 2로 설정해서 표시
        startPoint += 1

        if(startPoint < endPoint):
            array[pitch, startPoint:endPoint] = 1

    return array

def DatasetToScore(dataset : np.ndarray):
    '''ai가 출력한 데이터셋을 스코어 파일로 변환해서 리턴'''
    note_list = []

    indices = np.where(dataset == 2)
    coordinates = list(zip(indices[0], indices[1])) # 음이 시작하는 위기들 모두 찾기
    for coordinate in coordinates:
        coordinatePitch, coordinateStartPoint = coordinate
        endPoint = 1
        while(coordinateStartPoint + endPoint < 1200 and dataset[coordinatePitch][coordinateStartPoint + endPoint] == 1):
            endPoint += 1
        
        # 반복문 돌면서 복원한 값 집어넣기
        note_list.append((
            coordinateStartPoint / 40.0,
            endPoint / 40.0,
            int(coordinatePitch),
            127
        ))
    
    fields = [('onset_sec', 'f8'), ('duration_sec', 'f8'), ('pitch', 'i4'), ('velocity', 'i4')]
    note_array = np.array(note_list, dtype=fields)

    return pt.performance.PerformedPart.from_note_array(note_array)



    


## GAN 모델 학습시키기

### 1. 학습용 데이터셋들 준비

In [ ]:
import shutil

num = 1
save_dir = "processed_data"

if os.path.exists(save_dir):
    shutil.rmtree(save_dir)

os.makedirs(os.path.join(save_dir, "X"), exist_ok=True)
os.makedirs(os.path.join(save_dir, "Y"), exist_ok=True)

for filepath in file_list[:3]:
    performance = load_midi_from_virtual_folder(scarlatti_folder, filepath)
    performance = Score_Quantize(partituraPerformance=performance) # 퀀타이즈
    score = load_midi_from_virtual_folder(scarlatti_folder, filepath,target_type="music21")
    transposed_list = Score_TransposeToAllKeys(partituraPerformance=performance,music21Score=score) # 모든조로 전조
    for transposed_performance in transposed_list:
        slicedPerformances = Score_SliceByMeasures(partituraPerformance=transposed_performance) # 30초단위로 나눔
        for i in range(len(slicedPerformances)): # TODO 이거를 range(len())으로 고쳐서 splitpoint 설정하게 해야 함
            keyNum = PredictedKeyToNum(score)
            trebble, bass = Score_TrebleBassSeparation_Partitura(partituraPerformance=slicedPerformances[i], splitPoint = 60) # 오른손왼손 분리
            if trebble is None or len(trebble.note_array()) == 0:
                continue
            trebble = Score_MaskNotes(partituraTrebblePerformance=trebble)

            x = ScoreToDataset(trebble)
            y = ScoreToDataset(slicedPerformances[i])
            x = x.astype(np.uint8)
            y = y.astype(np.uint8)

            x_path =  os.path.join(save_dir, "X", f"sample_{num}.npz")
            y_path =  os.path.join(save_dir, "Y", f"sample_{num}.npz")
            np.savez_compressed(x_path, data=x)
            np.savez_compressed(y_path, data=y)

            num += 1


    

### 2. 모델 학습

In [ ]:
import torch

